In [18]:
import numpy as np
from gurobipy import *

In [19]:
import pandas as pd

data = {
    'Start': ['O', 'O', 'O', 'A', 'A', 'B', 'B', 'B', 'C', 'D', 'D', 'E'],
    'End':   ['A', 'B', 'C', 'B', 'D', 'C', 'D', 'E', 'E', 'E', 'T', 'T'],
    'Cost':  [2, 5, 4, 2, 7, 1, 4, 3, 4, 1, 5, 7]
}

df = pd.DataFrame(data)
print(df)

   Start End  Cost
0      O   A     2
1      O   B     5
2      O   C     4
3      A   B     2
4      A   D     7
5      B   C     1
6      B   D     4
7      B   E     3
8      C   E     4
9      D   E     1
10     D   T     5
11     E   T     7


In [20]:
pd.unique(df[['Start', 'End']].values.ravel())

array(['O', 'A', 'B', 'C', 'D', 'E', 'T'], dtype=object)

In [22]:
list_of_vertex = pd.unique(df[['Start', 'End']].values.ravel())
print(len(list_of_vertex))
num_of_edges = len(df)
print(num_of_edges)

7
12


In [23]:
# Create a Dictionary for the edges and costs
flow_dict = {}
for i in range(len(df)):
  flow_dict[
      df['Start'][i],
      df['End'][i]
      ] = df['Cost'][i]
print(flow_dict)

{('O', 'A'): np.int64(2), ('O', 'B'): np.int64(5), ('O', 'C'): np.int64(4), ('A', 'B'): np.int64(2), ('A', 'D'): np.int64(7), ('B', 'C'): np.int64(1), ('B', 'D'): np.int64(4), ('B', 'E'): np.int64(3), ('C', 'E'): np.int64(4), ('D', 'E'): np.int64(1), ('D', 'T'): np.int64(5), ('E', 'T'): np.int64(7)}


In [24]:
edges, costs = multidict(flow_dict)
print(edges)

<gurobi.tuplelist (12 tuples, 2 values each):
 ( O , A )
 ( O , B )
 ( O , C )
 ( A , B )
 ( A , D )
 ( B , C )
 ( B , D )
 ( B , E )
 ( C , E )
 ( D , E )
 ( D , T )
 ( E , T )
>


In [25]:
print(costs)

{('O', 'A'): np.int64(2), ('O', 'B'): np.int64(5), ('O', 'C'): np.int64(4), ('A', 'B'): np.int64(2), ('A', 'D'): np.int64(7), ('B', 'C'): np.int64(1), ('B', 'D'): np.int64(4), ('B', 'E'): np.int64(3), ('C', 'E'): np.int64(4), ('D', 'E'): np.int64(1), ('D', 'T'): np.int64(5), ('E', 'T'): np.int64(7)}


The TransformationWhen you run arcs, cost = multidict(flow_dict), the function "splits" that dictionary into two distinct parts:

arcs (The Index): This becomes a Gurobi tuplelist. It contains just the keys—the pairs of nodes representing your edges. You use this later to define your decision variables $x_{ij}$

cost (The Mapping): This becomes a dictionary-like object (a Gurobi tupledict) that maps each arc to its cost. You use this in your objective function $\sum d_{ij}x_{ij}$.

Quiz 1: Build the model for calculating shortest path

In [26]:
list_of_vertex

array(['O', 'A', 'B', 'C', 'D', 'E', 'T'], dtype=object)

In [27]:
qz1 = Model('Shortest Path')

# Add variables (edges)
x = {}
for edge in edges:
    x[edge] = qz1.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"{edge}")

# Set objective value
qz1.setObjective(quicksum(x[edge] * costs[edge] for edge in edges), GRB.MINIMIZE)

# Constraints
# Source Node
start_list = []
for edge in edges:
    if "O" == edge[0]:  # edges from the starting point
        start_list.append(x[edge])
qz1.addConstr(quicksum(start_list) == 1, name = "start_point")

# End Node
end_list = []
for edge in edges:
    if "T" == edge[1]:  # edges from the starting point
        end_list.append(x[edge])  # edge is a tuple, x[edge] is a variable
qz1.addConstr(quicksum(end_list) == 1, name = "end_point")

# Transshipment Node
trans_list = [x for x in list_of_vertex if x not in {"O", "T"}]
for vertex in trans_list:
    incoming_list = []
    outgoing_list = []
    for edge in edges:
        if vertex == edge[0]:
            incoming_list.append(x[edge])
        elif vertex == edge[1]:
            outgoing_list.append(x[edge])
        else:
            pass
    qz1.addConstr(quicksum(incoming_list) == quicksum(outgoing_list), name =f"incoming = outgoing")

# Call the solver
qz1.optimize()

# Print results
if qz1.status == GRB.OPTIMAL: # gp.GRB.OPTIMAL is a named constant: 2 optimal, 3 infeasible, 5 unbounded
    for var in qz1.getVars():
        print(var.varName, "=", var.x)
print(f"Objective value = {qz1.objVal}")

['A', 'B', 'C', 'D', 'E']
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5950X 16-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 7 rows, 12 columns and 24 nonzeros (Min)
Model fingerprint: 0xa5043857
Model has 12 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 7e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 3 rows and 6 columns
Presolve time: 0.01s
Presolved: 4 rows, 6 columns, 12 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    8.9960000e+00   2.004000e+00   0.000000e+00      0s
       2    1.3000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.300000000e+01
('O', 'A') = 1.0
('O', 'B') = 0.0
('O', 'C') =

Quiz 1: Build the model for calculating shortest path

In [33]:
edge_cap = costs.copy()
edge_cap[('T', 'O')] = 9999
print(edge_cap)

{('O', 'A'): np.int64(2), ('O', 'B'): np.int64(5), ('O', 'C'): np.int64(4), ('A', 'B'): np.int64(2), ('A', 'D'): np.int64(7), ('B', 'C'): np.int64(1), ('B', 'D'): np.int64(4), ('B', 'E'): np.int64(3), ('C', 'E'): np.int64(4), ('D', 'E'): np.int64(1), ('D', 'T'): np.int64(5), ('E', 'T'): np.int64(7), ('T', 'O'): 9999}


In [30]:
edges

<gurobi.tuplelist (12 tuples, 2 values each):
 ( O , A )
 ( O , B )
 ( O , C )
 ( A , B )
 ( A , D )
 ( B , C )
 ( B , D )
 ( B , E )
 ( C , E )
 ( D , E )
 ( D , T )
 ( E , T )
>

In [37]:
modi_edges = edges + [("T", "O")]
print(modi_edges)

<gurobi.tuplelist (13 tuples, 2 values each):
 ( O , A )
 ( O , B )
 ( O , C )
 ( A , B )
 ( A , D )
 ( B , C )
 ( B , D )
 ( B , E )
 ( C , E )
 ( D , E )
 ( D , T )
 ( E , T )
 ( T , O )
>


In [38]:
qz2 = Model('Maximum Flow')
# Add variables (edges)
x2 = {}
for edge in modi_edges:
    x2[edge] = qz2.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"{edge}")


# Set objective value
qz2.setObjective(-x2[("T","O")], GRB.MINIMIZE)

# Constraints
# # Source Node
# start_list = []
# for edge in edges:
#     if "O" == edge[0]:  # edges from the starting point
#         start_list.append(x[edge])
# qz1.addConstr(quicksum(start_list) == 1, name = "start_point")
#

# # End Node
# end_list = []
# for edge in edges:
#     if "T" == edge[1]:  # edges from the starting point
#         end_list.append(x[edge])  # edge is a tuple, x[edge] is a variable
# qz1.addConstr(quicksum(end_list) == 1, name = "end_point")

# Transshipment Node
# trans_list = [x for x in list_of_vertex if x not in {"O", "T"}]
for vertex in list_of_vertex:
    incoming_list = []
    outgoing_list = []
    for edge in modi_edges:
        if vertex == edge[0]:
            incoming_list.append(x2[edge])
        elif vertex == edge[1]:
            outgoing_list.append(x2[edge])
        else:
            pass
    qz2.addConstr(quicksum(incoming_list) == quicksum(outgoing_list), name =f"incoming = outgoing")

# Capacity Limits
for edge in modi_edges:
    qz2.addConstr(x2[edge] <= edge_cap[edge], name = f"Line capacity for {edge}")


# Call the solver
qz2.optimize()

# Print results
if qz2.status == GRB.OPTIMAL: # gp.GRB.OPTIMAL is a named constant: 2 optimal, 3 infeasible, 5 unbounded
    for var in qz2.getVars():
        print(var.varName, "=", var.x)
print(f"Objective value = {qz2.objVal}")

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5950X 16-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 20 rows, 13 columns and 39 nonzeros (Min)
Model fingerprint: 0xc63670f5
Model has 1 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+04]
Presolve removed 15 rows and 2 columns
Presolve time: 0.00s
Presolved: 5 rows, 11 columns, 18 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -1.2000000e+01   6.000000e+00   0.000000e+00      0s
       3   -1.1000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.01 seconds (0.00 work units)
Optimal objective -1.100000000e+01
('O', 'A') = 2.0
('O', 'B') = 5.0
('O', 'C') = 4.0
('A', 'B') = 2.0
('

Correct Rule (Ghouila-Houri Criterion)

A matrix A with entries in {0,1,−1} is totally unimodular if:

For every subset of rows, you can assign each row a sign +1 or −1 such that the signed sum of those rows has entries only in {−1,0,1}.